Multivariate Analysis : 
1. Small EDA
2. Pipeline creation
3. Comparing the ML models
4. Save the model for streamlit

In [ ]:
# importing requirements
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)

RAW = {
    "kaub_level":   "../data/raw/water_level_15min_kaub.csv",
    "bingen_level": "../data/raw/bingen_water_level15_min.csv",
    "discharge":    "../data/raw/kaub_water_discharge.csv",
}
WEATHER_HOURLY = "../data/processed/kaub_water_weather_hourly.csv"
WEATHER_DAILY  = "../data/processed/kaub_water_weather_daily.csv"

OUT_HOURLY = "../data/processed/kaub_hydro_weather_hourly.csv"
OUT_DAILY  = "../data/processed/kaub_hydro_weather_daily.csv"

In [ ]:
#Checking for the data
def load_raw(path: str, name: str) -> pd.Series:
    """Load and clean raw 15-minute water-level data from WSV."""

    # Read the raw CSV file using a semicolon as the separator
    d = pd.read_csv(path, sep=";")

    # Convert timestamps to datetime format
    d["timestamp"] = pd.to_datetime(d["timestamp"])

    # Convert water-level values to numeric and replace invalid values with NaN
    d["value"] = pd.to_numeric(d["value"], errors="coerce")

    # Generate a data quality report before removing duplicates
    report = {
        "rows": len(d),
        "duplicate timestamps": int(d["timestamp"].duplicated().sum()),
        "non-numeric values": int(d["value"].isna().sum()),
        "first": d["timestamp"].min(),
        "last": d["timestamp"].max(),
    }

    # Handle duplicate timestamps by calculating their mean water level
    # Sort chronologically and assign the station name to the Series
    s = d.groupby("timestamp")["value"].mean().sort_index().rename(name)

    # Record the number of observations after removing duplicate timestamps
    report["rows after dedup"] = len(s)

    # Display the data quality report for each station
    print(f"--- {name} ---")
    for k, v in report.items():
        print(f"  {k:22s}: {v}")

    # Return the cleaned water-level time series
    return s


# Load and clean raw water-level data for all stations defined in RAW
# Store the cleaned time series in a dictionary using station names as keys
series = {name: load_raw(path, name) for name, path in RAW.items()}

In [ ]:
# Create a dictionary to store duplicate timestamp information for each station
dup_check = {}

# Loop through all stations and their corresponding raw data files
for name, path in RAW.items():

    # Read the raw CSV file using a semicolon as the separator
    d = pd.read_csv(path, sep=";")

    # Convert timestamps to datetime format
    d["timestamp"] = pd.to_datetime(d["timestamp"])

    # Identify all rows with duplicate timestamps, including their first occurrence
    dups = d.loc[d["timestamp"].duplicated(keep=False), "timestamp"]

    # Summarize duplicate timestamp information for each station
    dup_check[name] = {

        # Count the total number of rows involved in duplicate timestamps
        "n duplicated rows": len(dups),

        # Count the number of unique dates affected by duplicate timestamps
        "distinct dates affected": dups.dt.date.nunique(),

        # Identify the months in which duplicate timestamps occur
        "months affected": sorted(dups.dt.month.unique().tolist()),

        # Identify the hours in which duplicate timestamps occur
        "hours affected": sorted(dups.dt.hour.unique().tolist()),

    }

# Convert the duplicate summary into a DataFrame for comparison across stations
pd.DataFrame(dup_check).T

In [ ]:
# Load the processed daily hydro-weather dataset
final_merged_df = pd.read_csv(
    "../data/processed/kaub_hydro_weather_daily.csv"
)

# Identify all numeric columns in the dataset
numeric_cols = final_merged_df.select_dtypes(include="number").columns

# Round numeric values to the nearest integer
# Convert them to nullable integers (Int64) to preserve missing values
final_merged_df[numeric_cols] = (
    final_merged_df[numeric_cols]
    .round()
    .astype("Int64")
)

# Display the first five rows of the updated dataset
final_merged_df.head()

In [ ]:
#final merged data
final_merged_df.head()

In [ ]:
#assigned to df 
df = final_merged_df

In [ ]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Select all numeric columns
#numeric_cols = df.select_dtypes(include="number").columns

# Round values and convert to integers
#df[numeric_cols] = df[numeric_cols].round(0).astype("Int64")

# Sort by date
df = df.sort_values("date").reset_index(drop=True)

# Display the DataFrame
df.head()

In [ ]:
# Count the number of days for each recorded number of available hourly observations
# Sort the results by the number of hours to identify days with missing or extra observations
df["hours_available"].value_counts().sort_index()

In [ ]:
# Select the main hydrological variables for correlation analysis
cols = [
    "water_level_mean",
    "bingen_level_mean",
    "discharge_mean"
]

# Calculate the Pearson correlation coefficients between the selected variables
# Identify the strength and direction of their linear relationships
df[cols].corr()

In [ ]:

# Set the figure size for the correlation heatmap
plt.figure(figsize=(7, 5))

# Create a heatmap to visualize correlations between hydrological variables
sns.heatmap(
    df[cols].corr(),  # Calculate Pearson correlation coefficients
    annot=True,       # Display correlation values inside each cell
    fmt=".2f",        # Show correlation values with two decimal places
    cmap="coolwarm",  # Use a color gradient to represent correlation strength
    vmin=-1,          # Set the minimum correlation value
    vmax=1            # Set the maximum correlation value
)

# Add a descriptive title to the heatmap
plt.title("Correlation of Hydrological Variables")

# Display the correlation heatmap
plt.show()

In [ ]:
# Calculate the correlation between Kaub water levels and lagged Bingen water levels
# Test lags from 0 to 7 days to examine the upstream-downstream relationship
for lag in range(0, 8):

    # Shift Bingen water levels by the specified number of days
    # Calculate the Pearson correlation with Kaub water levels
    corr = df["water_level_mean"].corr(
        df["bingen_level_mean"].shift(lag)
    )

    # Display the correlation coefficient for each lag
    print(f"Bingen lag {lag}: r = {corr:.3f}")

In [ ]:
# Calculate the correlation between Kaub water levels and lagged discharge values
# Test lags from 0 to 7 days to examine their temporal relationship
for lag in range(0, 8):

    # Shift discharge values by the specified number of days
    # Calculate the Pearson correlation with Kaub water levels
    corr = df["water_level_mean"].corr(
        df["discharge_mean"].shift(lag)
    )

    # Display the correlation coefficient for each lag
    print(f"Discharge lag {lag}: r = {corr:.3f}")

In [ ]:
# Calculate cumulative precipitation over different time windows
# Rolling sums capture the effect of recent rainfall on river water levels

# Calculate total precipitation over the current and previous 2 days
df["precip_3d"] = df["precipitation_sum"].rolling(3).sum()

# Calculate total precipitation over the current and previous 6 days
df["precip_7d"] = df["precipitation_sum"].rolling(7).sum()

# Calculate total precipitation over the current and previous 13 days
df["precip_14d"] = df["precipitation_sum"].rolling(14).sum()

# Calculate total precipitation over the current and previous 29 days
df["precip_30d"] = df["precipitation_sum"].rolling(30).sum()

In [ ]:
# Calculate the correlation between water levels and cumulative precipitation over different time windows
# Identify which rainfall period has the strongest linear relationship with Kaub water levels

rain_cols = [
    "precipitation_sum",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d"
]

df[rain_cols + ["water_level_mean"]].corr()["water_level_mean"]

In [ ]:
# Visualize the correlations among hydrological and weather variables
# Identify relationships with water levels and potential multicollinearity among features

features = [
    "water_level_mean",
    "temperature_mean",
    "humidity_mean",
    "precipitation_sum",
    "pressure_mean",
    "wind_speed_mean",
    "bingen_level_mean",
    "discharge_mean",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d"
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    df[features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Multivariate Correlation Matrix")

plt.show()

In [ ]:
#Fixing the target feature
TARGET = "water_level_mean"

In [ ]:
# Kaub water-level lags
df["level_lag_1"] = df["water_level_mean"].shift(1)
df["level_lag_2"] = df["water_level_mean"].shift(2)
df["level_lag_3"] = df["water_level_mean"].shift(3)
df["level_lag_7"] = df["water_level_mean"].shift(7)
df["level_lag_14"] = df["water_level_mean"].shift(14)

In [ ]:
# Calculate the daily change in Kaub water level compared to the previous day

df["level_change_1d"] = df["water_level_mean"].diff(1)

In [ ]:
# Create lagged Bingen water-level features for the previous 1, 2, and 3 days
# Capture the influence of historical upstream water levels on Kaub

df["bingen_lag_1"] = df["bingen_level_mean"].shift(1)

df["bingen_lag_2"] = df["bingen_level_mean"].shift(2)

df["bingen_lag_3"] = df["bingen_level_mean"].shift(3)

In [ ]:
# Create lagged discharge features for the previous 1, 2, and 3 days
# Capture the influence of historical river discharge on Kaub water levels

df["discharge_lag_1"] = df["discharge_mean"].shift(1)

df["discharge_lag_2"] = df["discharge_mean"].shift(2)

df["discharge_lag_3"] = df["discharge_mean"].shift(3)

In [ ]:
# Calculate the daily change in river discharge compared to the previous day

df["discharge_change_1d"] = df["discharge_mean"].diff(1)

In [ ]:
# Calculate cumulative precipitation over 3, 7, 14, and 30 days
# Capture the influence of recent rainfall on river water levels

df["precip_3d"] = df["precipitation_sum"].rolling(3).sum()

df["precip_7d"] = df["precipitation_sum"].rolling(7).sum()

df["precip_14d"] = df["precipitation_sum"].rolling(14).sum()

df["precip_30d"] = df["precipitation_sum"].rolling(30).sum()

In [ ]:
# Calculate the average temperature over 3 and 7 days
# Capture short-term temperature trends that may influence river water levels

df["temp_3d"] = df["temperature_mean"].rolling(3).mean()

df["temp_7d"] = df["temperature_mean"].rolling(7).mean()

In [ ]:
# Create cyclical day-of-year features using sine and cosine transformations
# Capture seasonal patterns while preserving the continuity between the end and start of each year

import numpy as np

df["day_of_year"] = df["date"].dt.dayofyear

df["doy_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["doy_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

In [ ]:
# Create target variables for forecasting Kaub water levels from 1 to 7 days ahead
# Shift water levels backward to align each future target with the current day's features

for h in range(1, 8):

    df[f"target_day_{h}"] = df["water_level_mean"].shift(-h)

In [ ]:
# Define the input features for multivariate water-level forecasting
# Include historical river conditions, weather patterns, and seasonal variations

FEATURES = [
    # Kaub history
    "level_lag_1",
    "level_lag_2",
    "level_lag_3",
    "level_lag_7",
    "level_lag_14",
    "level_change_1d",

    # Upstream Bingen
    "bingen_lag_1",
    "bingen_lag_2",
    "bingen_lag_3",

    # Discharge
    "discharge_lag_1",
    "discharge_lag_2",
    "discharge_lag_3",
    "discharge_change_1d",

    # Rainfall memory
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d",

    # Temperature
    "temp_3d",
    "temp_7d",

    # Seasonality
    "doy_sin",
    "doy_cos"
]

In [ ]:
# Define the target columns for predicting Kaub water levels from 1 to 7 days ahead

Target_cols = [
    f"target_day_{h}"
    for h in range(1, 8)
]

In [ ]:
# Calculate correlations between input features and future water levels for all seven forecast horizons
# Identify how the linear relationship between each feature and the target changes over time

TARGETS = [
    f"target_day_{h}"
    for h in range(1, 8)
]

corr_horizons = df[
    FEATURES + TARGETS
].corr().loc[FEATURES, TARGETS]

display(corr_horizons.round(3))

In [ ]:
# Heatmap : Correlation between engineered features
feature_corr = df[FEATURES].corr()

plt.figure(figsize=(16, 13))

sns.heatmap(
    feature_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title(
    "Correlation Matrix of Multivariate Features"
)

plt.tight_layout()
plt.show()

In [ ]:
model_df = df[
    ["date"] + FEATURES + TARGETS
].dropna().copy()

print(model_df.shape)
print(model_df["date"].min())
print(model_df["date"].max())

In [ ]:
# Print features
print("FEATURES:", FEATURES)

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nMissing features:")
print([col for col in FEATURES if col not in df.columns])

In [ ]:
# Kaub water level lag features
LAGS = [0, 1, 2, 3, 4, 5, 6, 7, 14, 21, 28]

for lag in LAGS:
    df[f"lag_{lag}"] = (
        df["water_level_mean"].shift(lag)
    )

In [ ]:
# Seven-day forecasting targets
TARGETS = []

for h in range(1, 8):

    target_col = f"target_day_{h}"

    df[target_col] = (
        df["water_level_mean"].shift(-h)
    )

    TARGETS.append(target_col)

In [ ]:
# Check whether all required columns exist
missing_cols = [
    col for col in FEATURES + TARGETS
    if col not in df.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)

else:
    # Calculate correlations
    corr_horizons = (
        df[FEATURES + TARGETS]
        .corr()
        .loc[FEATURES, TARGETS]
    )

    display(corr_horizons.round(3))

1. Univariate time-series

In [ ]:
# Prepare the univariate time-series dataset using historical Kaub water levels
# Create lag features and a target variable for forecasting water levels 7 days ahead

HORIZON = 7
H = HORIZON

LAGS = [0, 1, 2, 3, 4, 5, 6, 7, 14, 21, 28]

# Select Kaub daily mean water level
s = df["water_level_mean"]

data = pd.DataFrame(index=s.index)

for L in LAGS:
    data[f"lag_{L}"] = s.shift(L)

data["target"] = s.shift(-H)

FEATURES = [f"lag_{L}" for L in LAGS]

data.head(35)

In [ ]:
# Validate the alignment of lag features and the 7-day-ahead target
# Confirm that the current water level and future target correspond to the correct dates

i = 400

row = data.iloc[i]

assert row["lag_0"] == s.iloc[i]

assert row["target"] == s.iloc[i + H]

print(
    f"On {data.index[i]}: "
    f"today={row['lag_0']:.2f}, "
    f"target(+{H}d)={row['target']:.2f}"
)

In [ ]:
# Check the index type and values to verify the time-series date index

print(type(data.index))

print(data.index)

In [ ]:
# Set the date as a sorted datetime index for time-series analysis
# Select the daily mean Kaub water level as the target series

df["date"] = pd.to_datetime(df["date"])

df = df.set_index("date").sort_index()

s = df["water_level_mean"]

In [ ]:
# Define the forecast horizon and test period, then verify the dataset structure
# Check the date index, input features, data dimensions, and target availability

H = 7

TEST_YEARS = 3

print("Index type:", type(data.index))

print("Features:", FEATURES)

print("Data shape:", data.shape)

print("Target exists:", "target" in data.columns)

print("First date:", data.index.min())

print("Last date:", data.index.max())

In [ ]:
print(df.columns.tolist())
print(df.head(3))

In [ ]:
print(type(df.index))

In [ ]:
df.index = pd.to_datetime(df.index)
df = df.sort_index()

In [ ]:
# Prepare the multivariate dataset with a datetime index and selected input features
# Set the 7-day-ahead water level as the target and verify the dataset structure

df.index = pd.to_datetime(df.index)
df = df.sort_index()

FEATURES = [
    "lag_0", "lag_1", "lag_2", "lag_3",
    "lag_7", "lag_14", "lag_28",
    "precip_3d", "precip_7d", "precip_14d", "precip_30d",
    "temp_3d", "temp_7d",
    "bingen_lag_1", "bingen_lag_2", "bingen_lag_3",
    "doy_sin", "doy_cos"
]

data = df[FEATURES + ["target_day_7"]].copy()
data = data.rename(columns={"target_day_7": "target"})

print("df index:", type(df.index))
print("data index:", type(data.index))
print("Last date:", data.index.max())

In [ ]:
# Split the time-series data into training and test sets using the last 3 years for testing
# Apply a 7-day embargo to prevent overlapping forecast targets and verify temporal separation

H = 7

TEST_YEARS = 3

split_date = data.index.max() - pd.DateOffset(years=TEST_YEARS)

train = data.loc[:split_date - pd.Timedelta(days=H)].dropna()

test = data.loc[split_date + pd.Timedelta(days=1):].dropna()

X_train, y_train = train[FEATURES], train["target"]

X_test, y_test = test[FEATURES], test["target"]

print(f"train: {train.index.min().date()} → {train.index.max().date()} n={len(train)}")

print(f"test : {test.index.min().date()} → {test.index.max().date()} n={len(test)}")

assert train.index.max() + pd.Timedelta(days=H) < test.index.min(), "embargo violated"

In [ ]:
# Split the training data into model-fitting and validation sets, using the last 2 years for validation
# Apply a 7-day embargo to prevent target overlap and ensure temporal separation

VALID_YEARS = 2

validation_start = (
    train.index.max() - pd.DateOffset(years=VALID_YEARS)
    + pd.Timedelta(days=1)
)

train_fit = train.loc[
    :validation_start - pd.Timedelta(days=H + 1)
].copy()

validation = train.loc[
    validation_start:
].copy()

print(
    f"Model fitting: {train_fit.index.min().date()} : "
    f"{train_fit.index.max().date()} | n={len(train_fit)}"
)

print(
    f"Validation:    {validation.index.min().date()} : "
    f"{validation.index.max().date()} | n={len(validation)}"
)

assert (
    train_fit.index.max() + pd.Timedelta(days=H)
    < validation.index.min()
), "Validation embargo violated"

In [ ]:

# Actual water level 7 days ahead
y_val = validation["target"]

# Baseline prediction: today's observed water level
pred_baseline = validation["lag_0"].copy()

# Positive bias = overprediction; negative bias = underprediction
errors = pred_baseline - y_val

baseline_mae = errors.abs().mean()
baseline_rmse = np.sqrt((errors ** 2).mean())
baseline_bias = errors.mean()

baseline_results = pd.DataFrame([{
    "Model": "Persistence baseline",
    "Horizon (days)": H,
    "Samples": len(y_val),
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    #"R2" : r2_score(y_val, pred_baseline),
    "Bias": baseline_bias,
    "Skill vs persistence (MAE)": 0.0
}])

display(baseline_results.round(3))

In [ ]:
# Define feature groups to evaluate the contribution of additional weather and hydrological variables
# Compare different feature combinations to identify which inputs improve water-level forecasting

feature_groups = {
    "Kaub only": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28"
    ],

    "Kaub + precipitation": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28",
        "precip_3d", "precip_7d",
        "precip_14d", "precip_30d"
    ],

    "Kaub + precipitation + temperature": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28",
        "precip_3d", "precip_7d",
        "precip_14d", "precip_30d",
        "temp_3d", "temp_7d"
    ],

     #"Kaub + weather + Bingen + discharge": [
      #  "lag_0", "lag_1", "lag_2", "lag_3",
       # "lag_7", "lag_14", "lag_28",

        # Precipitation
        #"precip_3d", "precip_7d",
        #"precip_14d", "precip_30d",

        # Temperature
        #"temp_3d", "temp_7d",

        # Bingen water level
        #"bingen_lag_1", "bingen_lag_2",
        #"bingen_lag_3", "bingen_lag_7",

        # Kaub discharge
        #"discharge_lag_1", "discharge_lag_3",
        #"discharge_lag_7", "discharge_7d_mean"
    #],

    "All selected features": FEATURES
}

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

results = []

for group_name, columns in feature_groups.items():

    model = LinearRegression()

    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    predictions = model.predict(validation[columns])
    y_valid = validation["target"]

    mae = mean_absolute_error(y_valid, predictions)
    rmse = mean_squared_error(y_valid, predictions) ** 0.5
    r2 = r2_score(y_valid, predictions)
    #bias = np.mean(predictions - validation["target"]

    n = len(y_valid)
    p = len(columns)

    adjusted_r2 = (
        1 - (1 - r2) * (n - 1) / (n - p - 1)
        if n > p + 1 else float("nan")
    )

    bias = np.mean(predictions - y_valid.to_numpy())

    results.append({
        "Feature group": group_name,
        "Number of features": p,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "Bias":bias
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))

In [ ]:
# Evaluate the persistence baseline by assuming the water level remains unchanged after 7 days
# Calculate MAE and RMSE to compare baseline performance with the machine learning models

persistence_predictions = validation["lag_0"]

persistence_mae = mean_absolute_error(
    validation["target"],
    persistence_predictions
)

persistence_rmse = mean_squared_error(
    validation["target"],
    persistence_predictions
) ** 0.5

print(f"Persistence MAE:  {persistence_mae:.2f} cm")

print(f"Persistence RMSE: {persistence_rmse:.2f} cm")

In [ ]:
# Calculate the relative improvement in MAE compared with the persistence baseline
# Positive skill indicates better forecasting performance than persistence

results_df["Skill vs persistence"] = (
    1 - results_df["MAE"] / persistence_mae
)

display(results_df.round(3))

1. Modeling : LightGBM

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [ ]:
# Store results and trained models
lgbm_results = []
lgbm_models = {}

# Loop through each feature group
for group_name, columns in feature_groups.items():

    # Initialize LightGBM model
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=6,
        num_leaves=7,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=1.0,
        reg_alpha=0.2,
        reg_lambda=5.0,
        random_state=42,
        n_jobs=1,
        verbosity=-1
    )

    # Train model
    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    # Predict on validation data
    predictions = model.predict(
        validation[columns]
    )

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        validation["target"],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            validation["target"],
            predictions
        )
    )

    R2 = r2_score(
        validation["target"],
        predictions
    )

    bias = np.mean(predictions - y_valid.to_numpy())

    # Calculate skill vs persistence
    #skill = 1 - mae / persistence_mae

    # Store results
    lgbm_results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse,
        "R_squared": R2,
        "Bias" : bias
        #"Skill vs persistence": skill
    })

    # Store trained model
    lgbm_models[group_name] = model

# Create results DataFrame
lgbm_results_df = pd.DataFrame(lgbm_results)

# Display results
display(lgbm_results_df.round(3))

2. Random Forest :

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Actual targets and persistence baseline
y_valid = validation["target"].to_numpy()

persistence_mae = mean_absolute_error(
    y_valid, validation["lag_0"]
)

# Store results and trained models
rf_results = []
rf_models = {}

for group_name, columns in feature_groups.items():

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    )

    # Train on the earlier training period
    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    # Predict on the later validation period
    predictions = model.predict(validation[columns])

    # Validation metrics
    mae = mean_absolute_error(y_valid, predictions)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    r2 = r2_score(y_valid, predictions)

    # Positive bias = overprediction; negative = underprediction
    bias = np.mean(predictions - y_valid)

    skill = (
        1 - mae / persistence_mae
        if persistence_mae > 0 else np.nan
    )

    rf_results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse,
        "R_squared": r2,
        "Bias": bias,
        "Skill vs persistence": skill
    })

    rf_models[group_name] = model

rf_results_df = (
    pd.DataFrame(rf_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(rf_results_df.round(3))

In [ ]:
best_rf_group = rf_results_df.iloc[0]["Feature group"]
best_rf_model = rf_models[best_rf_group]

print("Best feature group:", best_rf_group)
print(f"Validation MAE: {rf_results_df.iloc[0]['MAE']:.3f} cm")

3. Modeling : XGBoost

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import pandas as pd
import numpy as np

# Train and evaluate XGBoost models using different feature groups
# Compare MAE, RMSE, R², Bias, and skill against the persistence baseline

xgb_results = []
xgb_models = {}

for group_name, columns in feature_groups.items():

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    predictions = model.predict(validation[columns])

    mae = mean_absolute_error(
        validation["target"],
        predictions
    )

    rmse = mean_squared_error(
        validation["target"],
        predictions
    ) ** 0.5

    R2 = r2_score(
        validation["target"],
        predictions
    )

    bias = np.mean(
        predictions - validation["target"]
    )

    xgb_results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse,
        "R_square": R2,
        "Bias": bias
    })

    xgb_models[group_name] = model

xgb_results_df = pd.DataFrame(xgb_results)

display(xgb_results_df.round(3))

In [ ]:
from sklearn.base import clone

# 1. Select the best feature group using validation results
best_group = xgb_results_df.loc[
    xgb_results_df["MAE"].idxmin(), "Feature group"
]

columns = xgb_models[best_group].get_booster().feature_names

# 2. Refit the selected model on all pre-test training data
final_xgb = clone(xgb_models[best_group])

final_xgb.fit(
    train[columns],
    train["target"]
)

# 3. Predict the held-out test data
y_test = test["target"]
test_predictions = final_xgb.predict(test[columns])

# Persistence baseline on exactly the same test rows
persistence_predictions = test["lag_0"]

test_mae = mean_absolute_error(y_test, test_predictions)
persistence_mae = mean_absolute_error(
    y_test, persistence_predictions
)

# 4. Display test metrics
test_results = pd.DataFrame([{
    "Model": "XGBoost",
    "Feature group": best_group,
    "Test samples": len(y_test),
    "MAE (cm)": test_mae,
    "RMSE (cm)": mean_squared_error(
        y_test, test_predictions
    ) ** 0.5,
    "R²": r2_score(y_test, test_predictions),
    "Bias (cm)": np.mean(
        test_predictions - y_test.to_numpy()
    ),
    "Persistence MAE (cm)": persistence_mae,
    "Skill vs persistence": (
        1 - test_mae / persistence_mae
        if persistence_mae > 0 else np.nan
    )
}])

display(test_results.round(3))

In [ ]:
# Generate predictions for the TEST data using the final model
columns = final_xgb.get_booster().feature_names
test_predictions = final_xgb.predict(test[columns])

# Build the error table using the same test rows
errors = pd.DataFrame({
    "Actual": test["target"],
    "Predicted": test_predictions,
    "Persistence": test["lag_0"]
}, index=test.index)

# Positive error = overprediction
errors["Error"] = errors["Predicted"] - errors["Actual"]
errors["Absolute error"] = errors["Error"].abs()

display(errors.head())

In [ ]:


# Define water conditions using ACTUAL water levels
low_threshold = 77
high_threshold = train["target"].quantile(0.90)

errors["Condition"] = np.select(
    [
        errors["Actual"] <= low_threshold,
        errors["Actual"] >= high_threshold
    ],
    ["Low water", "High water"],
    default="Normal water"
)

# Analyse all test days and each water condition
groups = [("All levels", errors)] + [
    (condition, errors.loc[errors["Condition"] == condition])
    for condition in ["Low water", "Normal water", "High water"]
]

results = []

for condition, subset in groups:
    if subset.empty:
        continue

    error = subset["Error"]  # predicted − actual
    baseline_error = subset["Persistence"] - subset["Actual"]

    mae = error.abs().mean()
    baseline_mae = baseline_error.abs().mean()

    results.append({
        "Condition": condition,
        "Days": len(subset),
        "MAE (cm)": mae,
        "RMSE (cm)": np.sqrt((error ** 2).mean()),
        "Bias (cm)": error.mean(),
        "Overprediction (%)": (error > 0).mean() * 100,
        "Persistence MAE (cm)": baseline_mae,
        "Skill vs persistence": (
            1 - mae / baseline_mae
            if baseline_mae > 0 else np.nan
        )
    })

error_summary = pd.DataFrame(results)
display(error_summary.round(2))

In [ ]:
# To see the MAE in each of the water condition
plot_data = error_summary.set_index("Condition").loc[
    ["Low water", "Normal water", "High water"],
    ["MAE (cm)", "Persistence MAE (cm)"]
]

plot_data.columns = ["XGBoost", "Persistence"]

ax = plot_data.plot.bar(
    figsize=(9, 5),
    color=["#0072B2", "#E69F00"],
    rot=0,
    width=0.7
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)

ax.set(
    title="Test Error by Water Condition",
    xlabel="Water condition",
    ylabel="MAE (cm) — lower is better"
)

ax.set_ylim(0, plot_data.to_numpy().max() * 1.2)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(title="Model")

plt.tight_layout()
plt.show()

In [ ]:
#Comparing XGBoost for Training and Validation data
group = "Kaub + precipitation + temperature"

model = xgb_models[group]
columns = feature_groups[group]

comparison = []

for name, subset in [
    ("Training", train_fit),
    ("Validation", validation)
]:
    actual = subset["target"].to_numpy()
    predicted = model.predict(subset[columns])
    errors = predicted - actual

    comparison.append({
        "Dataset": name,
        "Samples": len(actual),
        "MAE": np.abs(errors).mean(),
        "RMSE": np.sqrt((errors ** 2).mean()),
        "Bias": errors.mean()
    })

display(pd.DataFrame(comparison).round(3))

In [ ]:


# Select the feature group with the lowest validation MAE
best_group = xgb_results_df.loc[
    xgb_results_df["MAE"].idxmin(), "Feature group"
]

best_model = xgb_models[best_group]

# Use the exact features and order used during training
columns = best_model.get_booster().feature_names

y_true = validation["target"]
y_pred = best_model.predict(validation[columns])

# Residual = actual − predicted
residuals = y_true.to_numpy() - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals against predicted water level
axes[0].scatter(y_pred, residuals, alpha=0.5, s=25)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(
    title="Residuals vs Predicted Water Level",
    xlabel="Predicted water level (cm)",
    ylabel="Residual: actual − predicted (cm)"
)

# Residuals over time
axes[1].plot(validation.index, residuals, linewidth=1, alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(
    title="Residuals Over Time",
    xlabel="Forecast origin date",
    ylabel="Residual: actual − predicted (cm)"
)
axes[1].tick_params(axis="x", rotation=45)

for ax in axes:
    ax.grid(alpha=0.2)

fig.suptitle(f"Residual Analysis")
plt.tight_layout()
plt.show()

In [ ]:

# Select the model with the lowest validation MAE
best_group = xgb_results_df.loc[
    xgb_results_df["MAE"].idxmin(), "Feature group"
]

best_model = xgb_models[best_group]
columns = best_model.get_booster().feature_names

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5),
    sharex=True, sharey=True
)

for ax, (name, dataset) in zip(
    axes,
    [("Training", train_fit), ("Validation", validation)]
):
    # Predict using the same model and training features
    predictions = best_model.predict(dataset[columns])

    # Positive = underprediction; negative = overprediction
    residuals = dataset["target"].to_numpy() - predictions

    ax.scatter(
        predictions, residuals,
        alpha=0.35, s=20, color="steelblue"
    )

    ax.axhline(0, color="red", linestyle="--")
    ax.set_title(f"{name} Residuals")
    ax.set_xlabel("Predicted water level (cm)")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Residual: actual − predicted (cm)")

fig.suptitle(f"XGBoost — {best_group}")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Define thresholds using training targets only
low_threshold = train_fit["target"].quantile(0.10)
high_threshold = train_fit["target"].quantile(0.90)

print(f"Low-water threshold: {low_threshold:.1f} cm")
print(f"High-water threshold: {high_threshold:.1f} cm")

# 2. Predict validation water levels
columns = best_model.get_booster().feature_names

extreme_analysis = pd.DataFrame({
    "actual": validation["target"],
    "predicted": best_model.predict(validation[columns])
})

# Classify conditions using ACTUAL water levels
extreme_analysis["condition"] = np.select(
    [
        extreme_analysis["actual"] <= low_threshold,
        extreme_analysis["actual"] >= high_threshold
    ],
    ["Low water", "High water"],
    default="Normal range"
)

# Positive error means overprediction
extreme_analysis["error"] = (
    extreme_analysis["predicted"] - extreme_analysis["actual"]
)

# Persistence predicts the current level for the future target
extreme_analysis["persistence"] = validation["lag_0"]

# 3. Compare performance by water condition
results = []

for condition in ["Low water", "Normal range", "High water"]:
    subset = extreme_analysis[
        extreme_analysis["condition"] == condition
    ]

    if subset.empty:
        continue

    error = subset["error"]
    mae = error.abs().mean()

    baseline_mae = (
        subset["persistence"] - subset["actual"]
    ).abs().mean()

    results.append({
        "Condition": condition,
        "Days": len(subset),
        "MAE (cm)": mae,
        "RMSE (cm)": np.sqrt((error ** 2).mean()),
        "Bias (cm)": error.mean(),
        "Overprediction (%)": (error > 0).mean() * 100,
        "Persistence MAE (cm)": baseline_mae,
        "Skill vs persistence": (
            1 - mae / baseline_mae
            if baseline_mae > 0 else np.nan
        )
    })

extreme_results = pd.DataFrame(results)
display(extreme_results.round(2))

Tried to see the Polynomial Features 

In [ ]:

# Store transformed datasets and fitted transformers
poly_data = {}
poly_transformers = {}

for group_name, columns in feature_groups.items():

    # Degree 2: original features + squares + pairwise interactions
    # Exclude the constant column
    poly = PolynomialFeatures(
        degree=2,
        include_bias=False
    )

    # Fit on training data; apply the same transformation to validation
    X_train_poly = poly.fit_transform(train_fit[columns])
    X_valid_poly = poly.transform(validation[columns])

    feature_names = poly.get_feature_names_out(columns)

    # Convert to DataFrames, preserving dates and feature names
    poly_data[group_name] = {
        "train": pd.DataFrame(
            X_train_poly,
            columns=feature_names,
            index=train_fit.index
        ),
        "validation": pd.DataFrame(
            X_valid_poly,
            columns=feature_names,
            index=validation.index
        )
    }

    poly_transformers[group_name] = poly

    print(
        f"{group_name}: "
        f"{len(columns)} original → "
        f"{len(feature_names)} polynomial features"
    )

In [ ]:
group = "Kaub + precipitation + temperature"

print(poly_data[group]["train"].columns.tolist())
display(poly_data[group]["train"].head())

In [ ]:
for column in poly_data[group]["train"].columns:
    print(column)

In [ ]:
rows, columns = poly_data[group]["train"].shape

print("Number of rows:", rows)
print("Number of columns:", columns)

In [ ]:
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import Ridge
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import pandas as pd
# import numpy as np

poly_results = []
poly_models = {}

for group, datasets in poly_data.items():

    X_train_poly = datasets["train"]
    X_val_poly = datasets["validation"]

    # Match targets to the feature rows using their original date index
    y_train_poly = train_fit.loc[X_train_poly.index, "target"]
    y_val_poly = validation.loc[X_val_poly.index, "target"]

    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )

    model.fit(X_train_poly, y_train_poly)
    predictions = model.predict(X_val_poly)

    mae = mean_absolute_error(y_val_poly, predictions)
    rmse = np.sqrt(mean_squared_error(y_val_poly, predictions))
    r2 = r2_score(y_val_poly, predictions)
    bias = np.mean(predictions - y_val_poly.to_numpy())

    poly_results.append({
        "Feature group": group,
        "Number of features": X_train_poly.shape[1],
        "Validation MAE": mae,
        "Validation RMSE": rmse,
        "Validation R²": r2,
        "Validation Bias (cm)": bias
    })

    poly_models[group] = model

poly_results_df = (
    pd.DataFrame(poly_results)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

display(poly_results_df.round(3))

In [ ]:
# Polynomial Ridge results
poly_comparison = (
    poly_results_df[
        ["Feature group", "Validation MAE", "Validation RMSE"]
    ]
    .rename(columns={
        "Validation MAE": "MAE",
        "Validation RMSE": "RMSE"
    })
    .assign(Model="Polynomial Ridge")
)

# Original LightGBM results
lgbm_comparison = lgbm_results_df[
    ["Feature group", "MAE", "RMSE"]
].assign(Model="LightGBM")

# Random Forest results
rf_comparison = rf_results_df[
    ["Feature group", "MAE", "RMSE"]
].assign(Model="Random Forest")

# Combine model results
model_comparison = pd.concat(
    [poly_comparison, lgbm_comparison, rf_comparison],
    ignore_index=True
)

# Same validation persistence MAE for every model
model_comparison["Skill vs persistence (MAE)"] = (
    1 - model_comparison["MAE"] / persistence_mae
    if persistence_mae > 0 else np.nan
)

model_comparison = (
    model_comparison[
        ["Model", "Feature group", "MAE", "RMSE",
         "Skill vs persistence (MAE)"]
    ]
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(model_comparison.round(3))
print(f"Persistence validation MAE: {persistence_mae:.3f} cm")

In [ ]:
best_result = model_comparison.iloc[0]

print("Best model:", best_result["Model"])
print("Best feature group:", best_result["Feature group"])
print(f"Validation MAE: {best_result['MAE']:.3f} cm")

# Error Analysis :

In [ ]:
LOW_WATER_THRESHOLD = 77  # cm — replace with your chosen threshold

In [ ]:
import numpy as np
import pandas as pd

# Replace these with your fitted model and its feature list
model = xgb_models["Kaub + precipitation + temperature"]
columns = feature_groups["Kaub + precipitation + temperature"]

def evaluate_water_errors(model, data, columns, dataset_name, threshold):
    actual = data["target"].to_numpy()
    predicted = model.predict(data[columns])

    # Positive error means OVERPREDICTION
    error = predicted - actual

    results = []

    for condition, mask in {
        "All levels": np.ones(len(actual), dtype=bool),
        "Low water": actual < threshold,
    }.items():

        if not mask.any():
            continue

        errors = error[mask]

        results.append({
            "Dataset": dataset_name,
            "Condition": condition,
            "Samples": int(mask.sum()),
            "MAE": np.mean(np.abs(errors)),
            "RMSE": np.sqrt(np.mean(errors ** 2)),
            "Bias (predicted - actual)": np.mean(errors),
            "Overprediction (%)": 100 * np.mean(errors > 0),
            "Mean overprediction amount": np.maximum(errors, 0).mean(),
        })

    return pd.DataFrame(results)

results = pd.concat([
    evaluate_water_errors(
        model, train_fit, columns, "Training", LOW_WATER_THRESHOLD
    ),
    evaluate_water_errors(
        model, validation, columns, "Validation", LOW_WATER_THRESHOLD
    ),
], ignore_index=True)

display(results.round(2))

In [ ]:
# Final model to check in all water condition
group = "Kaub + precipitation + temperature"
model = xgb_models[group]  # Use your chosen fitted model
columns = feature_groups[group]

H = 7
LOW_WATER_THRESHOLD = 77  # Example: use your agreed threshold

errors = pd.DataFrame({
    "Actual": validation["target"],
    "Predicted": model.predict(validation[columns]),
    "Persistence": validation["lag_0"],
})

# Assumes the validation index is the forecast origin date
errors.index.name = "Forecast_origin"
errors["Target_date"] = errors.index + pd.Timedelta(days=H)

# Positive = overprediction; negative = underprediction
errors["Error"] = errors["Predicted"] - errors["Actual"]
errors["Absolute_error"] = errors["Error"].abs()
errors["Persistence_absolute_error"] = (
    errors["Persistence"] - errors["Actual"]
).abs()

errors["Low_water"] = errors["Actual"] < LOW_WATER_THRESHOLD
errors["Overprediction"] = errors["Error"] > 0

display(errors.head().round(2))

In [ ]:
#To make summary on overprediction 
def summarize_errors(frame):
    if frame.empty:
        return pd.Series({"Samples": 0})

    baseline_mae = frame["Persistence_absolute_error"].mean()
    model_mae = frame["Absolute_error"].mean()

    return pd.Series({
        "Samples": len(frame),
        "MAE": model_mae,
        "RMSE": np.sqrt((frame["Error"] ** 2).mean()),
        "Bias": frame["Error"].mean(),
        "Overprediction_pct": 100 * frame["Overprediction"].mean(),
        "P90_absolute_error": frame["Absolute_error"].quantile(0.90),
        "Persistence_MAE": baseline_mae,
        "Skill_pct": (
            100 * (1 - model_mae / baseline_mae)
            if baseline_mae > 0 else np.nan
        ),
    })

summary = pd.DataFrame({
    "All levels": summarize_errors(errors),
    "Low water": summarize_errors(errors.loc[errors["Low_water"]]),
    "Other levels": summarize_errors(errors.loc[~errors["Low_water"]]),
}).T

display(summary.round(2))

In [ ]:
#-day-ahead forecasts against actual water levels
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(
    errors["Target_date"], errors["Actual"],
    label="Actual", color="black"
)
axes[0].plot(
    errors["Target_date"], errors["Predicted"],
    label="XGBoost", alpha=0.8
)
axes[0].axhline(
    LOW_WATER_THRESHOLD,
    color="red", linestyle="--", label="Low-water threshold"
)
axes[0].set_ylabel("Water level (cm)")
axes[0].set_title("7-day-ahead forecasts against actual water levels")
axes[0].legend()

axes[1].plot(
    errors["Target_date"], errors["Error"],
    color="tab:orange"
)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_ylabel("Predicted − actual (cm)")
axes[1].set_xlabel("Target date")
axes[1].set_title("Above zero = overprediction")

plt.tight_layout()
plt.show()

In [ ]:
#worst water condition
worst_low_water = (
    errors.loc[errors["Low_water"] & errors["Overprediction"]]
    .nlargest(10, "Error")
)

display(
    worst_low_water[[
        "Target_date", "Actual", "Predicted",
        "Persistence", "Error"
    ]].round(2)
)

Save Model for streamlit

In [ ]:
# import joblib
# from pathlib import Path

# Select your trained model
best_group = "Kaub + precipitation + temperature"
model = xgb_models[best_group]

# Create a folder for saved models
Path("models").mkdir(parents=True, exist_ok=True)

# Save the model and its configuration
joblib.dump(
    {
        "model": model,
        "features": feature_groups[best_group],
        "horizon": H,
    },
    "models/kaub_xgboost.joblib"
)

print("Model saved successfully!")